# SSVEP EDA — Zhu et al. 2021 Wearable Dataset (Dry Electrodes)

This notebook characterizes the Zhu2021 dry-electrode SSVEP recordings, establishes calibration-free SNR/CCA/FBCCA baselines, selects a four-frequency command set on a subject-wise development cohort, and evaluates the fixed four-class system on held-out participants.

# 1. Dataset and experimental protocol

Zhu et al. (2021) wearable SSVEP dataset:

- **102 participants**
- **12 SSVEP targets**
- **8 posterior EEG channels**: `POz, PO3, PO4, PO5, PO6, Oz, O1, O2`
- **10 dry-electrode blocks** and **10 wet-electrode blocks** per participant
- publicly released sampling rate: **250 Hz**
- stored epoch length: **2.84 s = 710 samples**
- `.mat` data shape: `(8, 710, 2, 10, 12)`

The dimensions are:

```text
[channel, time, electrode_type, block, target_index]
```

The original EEG was acquired at **1000 Hz** and released after downsampling to **250 Hz**. Each experimental trial contained a **1 s visual cue**, **2 s visual stimulation**, and **1 s rest/feedback period**. Participants were instructed to fixate the cued target and avoid blinking during stimulation. Recordings were performed without electromagnetic shielding or acoustic isolation.

Only the **dry-electrode** recordings are used in this notebook.

## Official target-index mapping

| target index | frequency [Hz] | phase [π] |
|---:|---:|---:|
| 1 | 9.25 | 0.0 |
| 2 | 11.25 | 0.0 |
| 3 | 13.25 | 0.0 |
| 4 | 9.75 | 0.5 |
| 5 | 11.75 | 0.5 |
| 6 | 13.75 | 0.5 |
| 7 | 10.25 | 1.0 |
| 8 | 12.25 | 1.0 |
| 9 | 14.25 | 1.0 |
| 10 | 10.75 | 1.5 |
| 11 | 12.75 | 1.5 |
| 12 | 14.75 | 1.5 |

The fifth axis of the `.mat` files is therefore interpreted as a **target index**, not as an ascending-frequency axis.

# 2. Environment and configuration

In [ ]:
import warnings
warnings.simplefilter("default")

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from scipy.io import loadmat
from scipy.signal import (
    welch,
    cheby1,
    cheb1ord,
    butter,
    sosfiltfilt,
)
from sklearn.cross_decomposition import CCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

## 2.1 Global constants

In [ ]:
FS = 250

CHANNELS = ["POz", "PO3", "PO4", "PO5", "PO6", "Oz", "O1", "O2"]

TARGET_FREQS_BY_INDEX = np.array([
    9.25, 11.25, 13.25,
    9.75, 11.75, 13.75,
    10.25, 12.25, 14.25,
    10.75, 12.75, 14.75,
], dtype=float)

TARGET_PHASES_PI_BY_INDEX = np.array([
    0.0, 0.0, 0.0,
    0.5, 0.5, 0.5,
    1.0, 1.0, 1.0,
    1.5, 1.5, 1.5,
], dtype=float)

TARGET_PHASES_RAD_BY_INDEX = TARGET_PHASES_PI_BY_INDEX * np.pi
TARGET_FREQS_ALL = sorted(TARGET_FREQS_BY_INDEX.tolist())

DRY_ELECTRODE_INDEX = 0

PRE_STIM_SEC = 0.50
VISUAL_LATENCY_SEC = 0.14
STIM_DURATION_SEC = 2.00

ANALYSIS_START_SEC = PRE_STIM_SEC + VISUAL_LATENCY_SEC
ANALYSIS_DURATION_SEC = STIM_DURATION_SEC
ANALYSIS_START_SAMPLE = int(round(ANALYSIS_START_SEC * FS))
ANALYSIS_END_SAMPLE = ANALYSIS_START_SAMPLE + int(round(ANALYSIS_DURATION_SEC * FS))

stimulus_mapping = pd.DataFrame({
    "target_index": np.arange(1, 13),
    "frequency_hz": TARGET_FREQS_BY_INDEX,
    "phase_pi": TARGET_PHASES_PI_BY_INDEX,
})

display(stimulus_mapping)
print("Sorted target frequencies:", TARGET_FREQS_ALL)
print("Channels:", CHANNELS)
print("Analysis samples:", ANALYSIS_START_SAMPLE, ":", ANALYSIS_END_SAMPLE)

## 2.2 Analysis configuration

In [ ]:
DATA_DIR = Path("./Zhu2021_Wearable_SSVEP")
ALL_SUBJECTS = list(range(1, 103))

N_TEST_SUBJECTS = 20
SPLIT_RANDOM_STATE = 42

PSD_FILTER_BAND_HZ = (5.0, 90.0)
PSD_GRID_STEP_HZ = 0.25

N_HARMONICS = 5
FBCCA_N_SUBBANDS = 5
FBCCA_A = 2.0
FBCCA_B = 0.25

FBCCA_PASSBAND_LOW_HZ = np.array([6, 14, 22, 30, 38], dtype=float)
FBCCA_STOPBAND_LOW_HZ = np.array([4, 10, 16, 24, 32], dtype=float)
FBCCA_PASSBAND_HIGH_HZ = 90.0
FBCCA_STOPBAND_HIGH_HZ = 100.0

FBCCA_WEIGHTS = np.array([
    (idx + 1) ** (-FBCCA_A) + FBCCA_B
    for idx in range(FBCCA_N_SUBBANDS)
])

print("Subjects:", len(ALL_SUBJECTS))
print("Held-out subjects:", N_TEST_SUBJECTS)
print("Harmonics:", N_HARMONICS)
print("Subbands:", FBCCA_N_SUBBANDS)
print("FBCCA weights:", FBCCA_WEIGHTS)

## 2.3 FBCCA hyperparameters

The dry-electrode FBCCA configuration follows the Zhu2021 analysis and the classical M3-style filter-bank design.

The subband weights are

$$
w(n)=n^{-a}+b,
$$

with

$$
a=2.0,\qquad b=0.25.
$$

Five harmonics and five subbands are used. The first five passbands are

$$
[6,90],\ [14,90],\ [22,90],\ [30,90],\ [38,90]\ \mathrm{Hz},
$$

with corresponding lower stopband edges

$$
4,\ 10,\ 16,\ 24,\ 32\ \mathrm{Hz}.
$$

At $f_s=250\ \mathrm{Hz}$, the Nyquist frequency is $125\ \mathrm{Hz}$. The highest target frequency is $14.75\ \mathrm{Hz}$, so its fifth harmonic is

$$
5\cdot14.75=73.75\ \mathrm{Hz},
$$

which remains below the 90 Hz upper passband edge.

# 3. Helper functions

## 3.1 Filtering

In [ ]:
def butter_bandpass_filter_signal(signal, lowcut, highcut, fs, order=4):
    if not 0 < lowcut < highcut < fs / 2:
        raise ValueError("Band edges must satisfy 0 < lowcut < highcut < fs/2.")

    nyq = 0.5 * fs
    sos = butter(
        N=order,
        Wn=[lowcut / nyq, highcut / nyq],
        btype="bandpass",
        output="sos",
    )
    return sosfiltfilt(sos, signal)


def filter_trial_butter(trial, lowcut, highcut, fs, order=4):
    trial = np.asarray(trial, dtype=float)
    if trial.ndim != 2:
        raise ValueError("trial must have shape (n_channels, n_samples).")

    out = np.zeros_like(trial, dtype=float)
    for ch_idx in range(trial.shape[0]):
        out[ch_idx] = butter_bandpass_filter_signal(
            trial[ch_idx], lowcut, highcut, fs, order
        )
    return out


def filter_eeg_trials(trials, lowcut, highcut, fs, order=4):
    trials = np.asarray(trials, dtype=float)
    if trials.ndim != 3:
        raise ValueError("trials must have shape (n_trials, n_channels, n_samples).")

    out = np.zeros_like(trials, dtype=float)
    for trial_idx, trial in enumerate(trials):
        out[trial_idx] = filter_trial_butter(trial, lowcut, highcut, fs, order)
    return out


def design_fbcca_sos(subband_idx, fs):
    if not 0 <= subband_idx < FBCCA_N_SUBBANDS:
        raise ValueError("Invalid FBCCA subband index.")

    nyq = fs / 2.0
    if FBCCA_STOPBAND_HIGH_HZ >= nyq:
        raise ValueError("FBCCA upper stopband must be below the Nyquist frequency.")

    wp = [
        FBCCA_PASSBAND_LOW_HZ[subband_idx] / nyq,
        FBCCA_PASSBAND_HIGH_HZ / nyq,
    ]
    ws = [
        FBCCA_STOPBAND_LOW_HZ[subband_idx] / nyq,
        FBCCA_STOPBAND_HIGH_HZ / nyq,
    ]

    order, wn = cheb1ord(wp, ws, gpass=3, gstop=40)
    return cheby1(
        order,
        rp=0.5,
        Wn=wn,
        btype="bandpass",
        output="sos",
    )


def apply_sos_to_trial(trial, sos):
    trial = np.asarray(trial, dtype=float)
    if trial.ndim != 2:
        raise ValueError("trial must have shape (n_channels, n_samples).")

    out = np.zeros_like(trial, dtype=float)
    for ch_idx in range(trial.shape[0]):
        out[ch_idx] = sosfiltfilt(sos, trial[ch_idx])
    return out

## 3.2 PSD and SNR

In [ ]:
def compute_trial_mean_psd(trial, fs, nperseg=None, nfft=None):
    trial = np.asarray(trial, dtype=float)
    if trial.ndim != 2:
        raise ValueError("trial must have shape (n_channels, n_samples).")
    if fs <= 0:
        raise ValueError("fs must be positive.")

    if nperseg is None:
        nperseg = trial.shape[1]
    nperseg = min(int(nperseg), trial.shape[1])

    if nfft is None:
        nfft = max(nperseg, int(round(fs / PSD_GRID_STEP_HZ)))
    if nfft < nperseg:
        raise ValueError("nfft must be greater than or equal to nperseg.")

    psds = []
    freqs = None

    for channel in trial:
        freqs, psd = welch(
            channel,
            fs=fs,
            window="hann",
            nperseg=nperseg,
            noverlap=0,
            nfft=nfft,
            scaling="density",
        )
        psds.append(psd)

    return freqs, np.mean(psds, axis=0)


def compute_snr_at_freq(freqs, psd, target_freq, noise_width=2, skip_width=1):
    freqs = np.asarray(freqs, dtype=float)
    psd = np.asarray(psd, dtype=float)

    if freqs.ndim != 1 or psd.ndim != 1 or len(freqs) != len(psd):
        raise ValueError("freqs and psd must be one-dimensional arrays of equal length.")

    idx = int(np.argmin(np.abs(freqs - target_freq)))

    left = psd[idx - skip_width - noise_width : idx - skip_width]
    right = psd[idx + skip_width + 1 : idx + skip_width + noise_width + 1]

    if len(left) != noise_width or len(right) != noise_width:
        return np.nan, np.nan

    signal_psd = psd[idx]
    noise_psd = np.mean(np.concatenate([left, right]))

    if signal_psd <= 0 or noise_psd <= 0:
        return np.nan, np.nan

    ratio = signal_psd / noise_psd
    return float(ratio), float(10 * np.log10(ratio))

## 3.3 CCA references and correlation

In [ ]:
def generate_reference_signals(freq, n_samples, fs, n_harmonics=5):
    if freq <= 0 or fs <= 0 or n_samples <= 0 or n_harmonics < 1:
        raise ValueError("Invalid CCA reference parameters.")
    if n_harmonics * freq >= fs / 2:
        raise ValueError("Highest reference harmonic must be below Nyquist.")

    t = np.arange(n_samples, dtype=float) / fs
    refs = []

    for harmonic in range(1, n_harmonics + 1):
        refs.append(np.sin(2 * np.pi * harmonic * freq * t))
        refs.append(np.cos(2 * np.pi * harmonic * freq * t))

    return np.vstack(refs)


def get_cca_correlation(data, ref):
    data = np.asarray(data, dtype=float)
    ref = np.asarray(ref, dtype=float)

    if data.ndim != 2 or ref.ndim != 2:
        raise ValueError("data and ref must be two-dimensional arrays.")
    if data.shape[1] != ref.shape[1]:
        raise ValueError("EEG and reference signals must have the same number of samples.")
    if not np.all(np.isfinite(data)) or not np.all(np.isfinite(ref)):
        raise ValueError("CCA input contains NaN or Inf.")

    X = data.T
    Y = ref.T

    cca = CCA(n_components=1, max_iter=1000, scale=True)
    X_c, Y_c = cca.fit_transform(X, Y)

    corr = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
    if not np.isfinite(corr):
        return 0.0

    return float(abs(corr))

## 3.4 Classifiers

In [ ]:
def cca_predict(
    trials,
    true_labels,
    target_freqs,
    fs,
    filter_band=(6, 90),
    n_harmonics=5,
    metadata=None,
):
    trials = np.asarray(trials, dtype=float)
    true_labels = np.asarray(true_labels, dtype=float)

    if trials.ndim != 3:
        raise ValueError("trials must have shape (n_trials, n_channels, n_samples).")
    if len(trials) != len(true_labels):
        raise ValueError("Number of trials and labels must match.")
    if metadata is not None and len(metadata) != len(trials):
        raise ValueError("metadata must contain one row per trial.")

    n_samples = trials.shape[2]
    references = {
        float(freq): generate_reference_signals(
            float(freq), n_samples, fs, n_harmonics
        )
        for freq in target_freqs
    }

    rows = []

    for trial_idx, trial in enumerate(trials):
        filtered = filter_trial_butter(
            trial,
            lowcut=filter_band[0],
            highcut=filter_band[1],
            fs=fs,
            order=4,
        )

        scores = {
            float(freq): get_cca_correlation(
                filtered,
                references[float(freq)],
            )
            for freq in target_freqs
        }

        pred = max(scores, key=scores.get)

        row = {
            "trial": trial_idx,
            "true_label": float(true_labels[trial_idx]),
            "predicted_label": float(pred),
            "cca_score": float(scores[pred]),
        }

        if metadata is not None:
            for col in ["subject", "block", "target_index"]:
                row[col] = metadata.iloc[trial_idx][col]

        for freq, score in scores.items():
            row[f"CCA_{freq:.2f}"] = float(score)

        rows.append(row)

    return pd.DataFrame(rows)


def fbcca_predict(
    trials,
    true_labels,
    target_freqs,
    fs,
    n_harmonics=5,
    n_subbands=5,
    a=2.0,
    b=0.25,
    metadata=None,
    show_progress=True,
):
    trials = np.asarray(trials, dtype=float)
    true_labels = np.asarray(true_labels, dtype=float)

    if trials.ndim != 3:
        raise ValueError("trials must have shape (n_trials, n_channels, n_samples).")
    if len(trials) != len(true_labels):
        raise ValueError("Number of trials and labels must match.")
    if metadata is not None and len(metadata) != len(trials):
        raise ValueError("metadata must contain one row per trial.")
    if not 1 <= n_subbands <= FBCCA_N_SUBBANDS:
        raise ValueError(
            f"n_subbands must be between 1 and {FBCCA_N_SUBBANDS}."
        )

    n_samples = trials.shape[2]
    references = {
        float(freq): generate_reference_signals(
            float(freq), n_samples, fs, n_harmonics
        )
        for freq in target_freqs
    }

    sos_bank = [
        design_fbcca_sos(subband_idx, fs)
        for subband_idx in range(n_subbands)
    ]

    weights = np.array([
        (idx + 1) ** (-a) + b
        for idx in range(n_subbands)
    ])

    rows = []
    progress_every = max(1, len(trials) // 20)

    for trial_idx, trial in enumerate(trials):
        filtered_subbands = [
            apply_sos_to_trial(trial, sos)
            for sos in sos_bank
        ]

        scores = {}

        for target_freq in target_freqs:
            rho_sq = []

            for filtered in filtered_subbands:
                rho = get_cca_correlation(
                    filtered,
                    references[float(target_freq)],
                )
                rho_sq.append(rho ** 2)

            scores[float(target_freq)] = float(
                np.dot(weights, np.asarray(rho_sq))
            )

        pred = max(scores, key=scores.get)

        row = {
            "trial": trial_idx,
            "true_label": float(true_labels[trial_idx]),
            "predicted_label": float(pred),
            "fbcca_score": float(scores[pred]),
        }

        if metadata is not None:
            for col in ["subject", "block", "target_index"]:
                row[col] = metadata.iloc[trial_idx][col]

        for freq, score in scores.items():
            row[f"FBCCA_{freq:.2f}"] = float(score)

        rows.append(row)

        if show_progress and (
            (trial_idx + 1) % progress_every == 0
            or trial_idx + 1 == len(trials)
        ):
            print(
                f"\rFBCCA: {trial_idx + 1}/{len(trials)} trials",
                end="",
                flush=True,
            )

    if show_progress:
        print()

    return pd.DataFrame(rows)

## 3.5 Evaluation helpers

In [ ]:
def format_frequency_labels(values):
    return (
        pd.Series(values)
        .astype(float)
        .round(2)
        .map(lambda x: f"{x:.2f}")
    )


def ordered_frequency_labels(*series):
    numeric = sorted({
        float(value)
        for s in series
        for value in pd.Series(s).astype(float).tolist()
    })
    return [f"{value:.2f}" for value in numeric]


def safe_accuracy(df):
    return accuracy_score(
        format_frequency_labels(df["true_label"]),
        format_frequency_labels(df["predicted_label"]),
    )


def safe_balanced_accuracy(df):
    return balanced_accuracy_score(
        format_frequency_labels(df["true_label"]),
        format_frequency_labels(df["predicted_label"]),
    )


def print_metrics(df, name):
    y_true = format_frequency_labels(df["true_label"])
    y_pred = format_frequency_labels(df["predicted_label"])
    labels = ordered_frequency_labels(df["true_label"], df["predicted_label"])

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    print(f"{name} accuracy: {acc:.2%}")
    print(f"{name} balanced accuracy: {bacc:.2%}")
    print()
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            zero_division=0,
        )
    )


def plot_confusion(df, title):
    y_true = format_frequency_labels(df["true_label"])
    y_pred = format_frequency_labels(df["predicted_label"])
    labels = ordered_frequency_labels(df["true_label"], df["predicted_label"])

    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    disp.plot(values_format=".2f")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def per_class_accuracy(df):
    tmp = df.copy()
    tmp["correct"] = np.isclose(
        tmp["true_label"].astype(float),
        tmp["predicted_label"].astype(float),
        atol=0.01,
    )
    return tmp.groupby("true_label")["correct"].mean().sort_index()


def evaluate_four_class_subset(fbcca_score_df, subset):
    subset = tuple(float(x) for x in subset)

    mask = fbcca_score_df["true_label"].astype(float).isin(subset)
    df = fbcca_score_df.loc[mask].copy()

    score_cols = [f"FBCCA_{freq:.2f}" for freq in subset]
    score_matrix = df[score_cols].to_numpy()

    pred_idx = np.argmax(score_matrix, axis=1)
    preds = np.asarray(subset, dtype=float)[pred_idx]

    y_true = format_frequency_labels(df["true_label"])
    y_pred = format_frequency_labels(preds)

    spacing = min(
        abs(a - b)
        for i, a in enumerate(subset)
        for b in subset[i + 1:]
    )

    return {
        "frequencies": subset,
        "min_spacing_hz": spacing,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "n_trials": len(df),
    }

# 4. Data loading and integrity checks

The loader maps each target index to the official stimulation frequency and phase, selects dry EEG, extracts a 2 s response window beginning 140 ms after stimulus onset, and returns data in `(trials, channels, samples)` format.

In [ ]:
def subject_file(subject, data_dir=None):
    if data_dir is None:
        data_dir = DATA_DIR
    return Path(data_dir) / f"S{subject:03d}.mat"


def load_zhu2021_subject(
    subject,
    data_dir=None,
    dry_index=DRY_ELECTRODE_INDEX,
    analysis_start=ANALYSIS_START_SAMPLE,
    analysis_end=ANALYSIS_END_SAMPLE,
):
    if data_dir is None:
        data_dir = DATA_DIR

    path = subject_file(subject, data_dir)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Download and extract the Zhu2021 subject files first."
        )

    mat = loadmat(path)
    if "data" not in mat:
        raise KeyError(f"{path.name} does not contain a 'data' matrix.")

    data = np.asarray(mat["data"], dtype=float)

    expected_shape = (8, 710, 2, 10, 12)
    if data.shape != expected_shape:
        raise ValueError(
            f"Unexpected shape for {path.name}: {data.shape}; "
            f"expected {expected_shape}."
        )

    dry = data[:, :, dry_index, :, :]

    trials = []
    labels = []
    rows = []

    for block_idx in range(dry.shape[2]):
        for target_idx in range(dry.shape[3]):
            freq = float(TARGET_FREQS_BY_INDEX[target_idx])
            phase_pi = float(TARGET_PHASES_PI_BY_INDEX[target_idx])

            epoch = dry[
                :,
                analysis_start:analysis_end,
                block_idx,
                target_idx,
            ]

            trials.append(epoch)
            labels.append(freq)
            rows.append({
                "subject": int(subject),
                "block": int(block_idx + 1),
                "target_index": int(target_idx + 1),
                "frequency": freq,
                "phase_pi": phase_pi,
                "phase_rad": phase_pi * np.pi,
                "electrode_type": "dry",
            })

    return (
        np.stack(trials),
        np.asarray(labels, dtype=float),
        pd.DataFrame(rows),
    )


def load_zhu2021_subjects(subjects, data_dir=None):
    if data_dir is None:
        data_dir = DATA_DIR

    X_parts, y_parts, meta_parts = [], [], []

    for subject in subjects:
        X_s, y_s, meta_s = load_zhu2021_subject(subject, data_dir)
        X_parts.append(X_s)
        y_parts.append(y_s)
        meta_parts.append(meta_s)

    return (
        np.concatenate(X_parts, axis=0),
        np.concatenate(y_parts, axis=0),
        pd.concat(meta_parts, ignore_index=True),
    )


def check_subject_files(subjects, data_dir=None):
    if data_dir is None:
        data_dir = DATA_DIR

    missing = [
        subject_file(subject, data_dir)
        for subject in subjects
        if not subject_file(subject, data_dir).exists()
    ]

    if missing:
        preview = "\n".join(str(path) for path in missing[:20])
        extra = ""
        if len(missing) > 20:
            extra = f"\n... and {len(missing) - 20} more missing files."

        raise FileNotFoundError(
            f"Missing {len(missing)} of {len(subjects)} required subject files.\n"
            f"First missing files:\n{preview}{extra}\n\n"
            f"DATA_DIR resolves to:\n{Path(data_dir).resolve()}"
        )

    print(f"All {len(subjects)} requested subject files are present.")

In [ ]:
check_subject_files(ALL_SUBJECTS)

X_all, y_all, meta_all = load_zhu2021_subjects(ALL_SUBJECTS)

print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print("Subjects loaded:", meta_all["subject"].nunique())

assert len(X_all) == len(y_all) == len(meta_all)
assert X_all.ndim == 3
assert X_all.shape[1] == len(CHANNELS)
assert X_all.shape[2] == int(round(ANALYSIS_DURATION_SEC * FS))
assert meta_all["subject"].nunique() == len(ALL_SUBJECTS)
assert set(meta_all["subject"].unique()) == set(ALL_SUBJECTS)
assert np.all(np.isfinite(X_all))
assert np.all(np.isfinite(y_all))
assert np.allclose(
    np.sort(np.unique(y_all.astype(float))),
    np.sort(np.asarray(TARGET_FREQS_ALL, dtype=float)),
)
assert np.allclose(
    y_all.astype(float),
    meta_all["frequency"].to_numpy(dtype=float),
)

trials_per_subject = meta_all.groupby("subject").size()
assert (trials_per_subject == 120).all()

first_block = (
    meta_all[
        (meta_all["subject"] == ALL_SUBJECTS[0]) &
        (meta_all["block"] == 1)
    ][["target_index", "frequency", "phase_pi"]]
    .sort_values("target_index")
    .reset_index(drop=True)
)

assert len(first_block) == 12
assert np.array_equal(
    first_block["target_index"].to_numpy(),
    np.arange(1, 13),
)
assert np.allclose(
    first_block["frequency"].to_numpy(dtype=float),
    TARGET_FREQS_BY_INDEX,
)
assert np.allclose(
    first_block["phase_pi"].to_numpy(dtype=float),
    TARGET_PHASES_PI_BY_INDEX,
)

print("Trials per subject:", trials_per_subject.iloc[0])
print("Target-index mapping check: OK")
print("Dataset integrity checks: OK")

# 5. Subject-wise evaluation split

Participants are split before any data-driven four-class selection. The **development cohort** is used for EDA, baseline analysis, and frequency-subset selection. The **held-out cohort** is reserved for the final four-class evaluation.

In [ ]:
subjects = np.asarray(ALL_SUBJECTS, dtype=int)

selection_subjects, test_subjects = train_test_split(
    subjects,
    test_size=N_TEST_SUBJECTS,
    random_state=SPLIT_RANDOM_STATE,
    shuffle=True,
)

selection_subjects = np.sort(selection_subjects)
test_subjects = np.sort(test_subjects)

selection_mask = meta_all["subject"].isin(selection_subjects).to_numpy()
test_mask = meta_all["subject"].isin(test_subjects).to_numpy()

X_selection = X_all[selection_mask]
y_selection = y_all[selection_mask]
meta_selection = meta_all.loc[selection_mask].reset_index(drop=True)

X_test = X_all[test_mask]
y_test = y_all[test_mask]
meta_test = meta_all.loc[test_mask].reset_index(drop=True)

assert set(selection_subjects).isdisjoint(set(test_subjects))
assert len(selection_subjects) + len(test_subjects) == len(ALL_SUBJECTS)
assert len(X_selection) == len(y_selection) == len(meta_selection)
assert len(X_test) == len(y_test) == len(meta_test)

print("Development subjects:", len(selection_subjects))
print("Held-out subjects:", len(test_subjects))
print("Development trials:", len(y_selection))
print("Held-out trials:", len(y_test))

# 6. Development-set exploratory analysis

In [ ]:
dataset_summary = pd.DataFrame({
    "property": [
        "Electrode type",
        "Channels",
        "Sampling rate",
        "Analysis window",
        "Targets",
        "Total subjects",
        "Development subjects",
        "Held-out subjects",
        "Total dry trials",
    ],
    "value": [
        "dry",
        ", ".join(CHANNELS),
        f"{FS} Hz",
        f"{ANALYSIS_DURATION_SEC:.2f} s",
        len(TARGET_FREQS_ALL),
        len(ALL_SUBJECTS),
        len(selection_subjects),
        len(test_subjects),
        len(y_all),
    ],
})

display(dataset_summary)

class_counts_selection = pd.Series(y_selection).value_counts().sort_index()
display(class_counts_selection.rename("development_trials").to_frame())

## 6.1 Example EEG trial

In [ ]:
example_idx = 0
example_trial = X_selection[example_idx]
time = np.arange(example_trial.shape[1], dtype=float) / FS

assert example_trial.shape[0] == len(CHANNELS)

plt.figure(figsize=(12, 5))

offset = 0.0
for ch_idx, channel in enumerate(CHANNELS):
    signal = example_trial[ch_idx]
    scale = np.nanstd(signal)
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0

    plt.plot(time, signal / scale + offset, label=channel)
    offset += 5.0

plt.xlabel("Time within analysis window [s]")
plt.ylabel("Normalized channel traces — vertically offset")
plt.title(f"Example dry-electrode trial — {y_selection[example_idx]:.2f} Hz")
plt.yticks([])
plt.legend(loc="upper right", ncol=2)
plt.tight_layout()
plt.show()

# 7. Spectral analysis

PSD/SNR analysis uses a broad **5–90 Hz** Butterworth-filtered copy of the development data. The 2 s segment contains 500 samples. Welch PSD is evaluated on a **0.25 Hz grid** using zero-padding to `nfft = 1000`, so all Zhu2021 target frequencies are represented explicitly. Zero-padding refines the evaluation grid but does not increase the true spectral resolution of the 2 s recording.

In [ ]:
X_selection_filtered = filter_eeg_trials(
    X_selection,
    lowcut=PSD_FILTER_BAND_HZ[0],
    highcut=PSD_FILTER_BAND_HZ[1],
    fs=FS,
    order=4,
)

print("Development data:", X_selection.shape)
print("Filtered development data:", X_selection_filtered.shape)

## 7.1 Example PSD

In [ ]:
freqs, psd = compute_trial_mean_psd(
    X_selection_filtered[example_idx],
    fs=FS,
    nperseg=X_selection_filtered.shape[2],
)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs, psd)
plt.axvline(y_selection[example_idx], linestyle="--")
plt.xlim(5, 90)
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD")
plt.title(f"2-s PSD — true target {y_selection[example_idx]:.2f} Hz")
plt.tight_layout()
plt.show()

## 7.2 Class-wise median PSD

In [ ]:
plt.figure(figsize=(12, 6))

for target_freq in TARGET_FREQS_ALL:
    idxs = np.where(np.isclose(y_selection, target_freq))[0]

    class_psds = []
    for idx in idxs:
        f, p = compute_trial_mean_psd(
            X_selection_filtered[idx],
            fs=FS,
            nperseg=X_selection_filtered.shape[2],
        )
        class_psds.append(p)

    median_psd = np.median(np.vstack(class_psds), axis=0)
    plt.semilogy(f, median_psd, label=f"{target_freq:.2f} Hz")

plt.xlim(5, 90)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Median PSD")
plt.title("Class-wise median PSD — development subjects — dry EEG")
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

A prominent component around **50 Hz** may be visible because the recordings were collected without electromagnetic shielding. No 50 Hz notch is applied in the main pipeline because several SSVEP harmonics lie close to the line frequency and the FBCCA benchmark retains harmonic information up to 90 Hz.

# 8. Development-set baselines

## 8.1 Local spectral SNR baseline

In [ ]:
def build_snr_dataframe(trials, true_labels, target_freqs, fs, metadata=None):
    if len(trials) != len(true_labels):
        raise ValueError("Number of trials and labels must match.")
    if metadata is not None and len(metadata) != len(trials):
        raise ValueError("metadata must contain one row per trial.")

    rows = []

    for trial_idx, trial in enumerate(trials):
        freqs, trial_psd = compute_trial_mean_psd(
            trial,
            fs=fs,
            nperseg=trial.shape[1],
        )

        row = {
            "trial": trial_idx,
            "true_label": float(true_labels[trial_idx]),
        }

        if metadata is not None:
            for col in ["subject", "block", "target_index"]:
                row[col] = metadata.iloc[trial_idx][col]

        for freq in target_freqs:
            snr, snr_db = compute_snr_at_freq(
                freqs,
                trial_psd,
                float(freq),
            )
            row[f"SNR_{freq:.2f}"] = snr
            row[f"SNRdB_{freq:.2f}"] = snr_db

        rows.append(row)

    out = pd.DataFrame(rows)

    score_cols = [f"SNR_{freq:.2f}" for freq in target_freqs]
    out["predicted_label"] = (
        out[score_cols]
        .idxmax(axis=1)
        .str.replace("SNR_", "", regex=False)
        .astype(float)
    )

    return out


snr_selection_12 = build_snr_dataframe(
    X_selection_filtered,
    y_selection,
    TARGET_FREQS_ALL,
    FS,
    meta_selection,
)

print_metrics(
    snr_selection_12,
    "SNR baseline — 12 classes — development subjects",
)

## 8.2 Standard CCA baseline

In [ ]:
cca_selection_12 = cca_predict(
    X_selection,
    y_selection,
    TARGET_FREQS_ALL,
    FS,
    filter_band=(6, 90),
    n_harmonics=N_HARMONICS,
    metadata=meta_selection,
)

print_metrics(
    cca_selection_12,
    "CCA — 12 classes — development subjects",
)

## 8.3 FBCCA baseline

In [ ]:
fbcca_selection_12 = fbcca_predict(
    X_selection,
    y_selection,
    TARGET_FREQS_ALL,
    FS,
    n_harmonics=N_HARMONICS,
    n_subbands=FBCCA_N_SUBBANDS,
    a=FBCCA_A,
    b=FBCCA_B,
    metadata=meta_selection,
)

print_metrics(
    fbcca_selection_12,
    "FBCCA — 12 classes — development subjects",
)

display(per_class_accuracy(fbcca_selection_12))

# 9. Four-class frequency selection

All $\binom{12}{4}=495$ four-frequency subsets are evaluated **only on the development cohort**. For each subset, the existing FBCCA scores are restricted to the four candidate frequencies and each development trial is reclassified among those four candidates. The held-out participants do not contribute to this selection.

In [ ]:
selection_subset_results = pd.DataFrame([
    evaluate_four_class_subset(fbcca_selection_12, subset)
    for subset in combinations(TARGET_FREQS_ALL, 4)
])

selection_subset_results = selection_subset_results.sort_values(
    ["balanced_accuracy", "accuracy", "min_spacing_hz"],
    ascending=[False, False, False],
).reset_index(drop=True)

SELECTED_4_FREQS = list(
    selection_subset_results.iloc[0]["frequencies"]
)

print("Selected frequencies:", SELECTED_4_FREQS)
print(
    "Development balanced accuracy:",
    f"{selection_subset_results.iloc[0]['balanced_accuracy']:.2%}",
)

display(selection_subset_results.head(20))

## 9.1 Selected-frequency diagnostics

In [ ]:
selected_diagnostics = pd.DataFrame({
    "frequency": SELECTED_4_FREQS,
    "12class_FBCCA_accuracy_development": [
        np.mean(
            np.isclose(
                fbcca_selection_12.loc[
                    np.isclose(fbcca_selection_12["true_label"], freq),
                    "predicted_label",
                ],
                freq,
                atol=0.01,
            )
        )
        for freq in SELECTED_4_FREQS
    ],
    "median_SNR_dB_development": [
        snr_selection_12.loc[
            np.isclose(snr_selection_12["true_label"], freq),
            f"SNRdB_{freq:.2f}",
        ].median()
        for freq in SELECTED_4_FREQS
    ],
})

display(selected_diagnostics)

# 10. Final evaluation on held-out participants

The four frequencies are fixed before the held-out cohort is evaluated. Only trials belonging to the selected four classes are retained, and each held-out trial is classified among those four candidates.

In [ ]:
test_4_mask = np.isin(y_test, SELECTED_4_FREQS)

X_test_4 = X_test[test_4_mask]
y_test_4 = y_test[test_4_mask]
meta_test_4 = meta_test.loc[test_4_mask].reset_index(drop=True)

assert meta_test_4["subject"].nunique() == len(test_subjects)
assert len(y_test_4) == len(test_subjects) * 10 * 4

print("Held-out subjects:", meta_test_4["subject"].nunique())
print("Held-out four-class trials:", len(y_test_4))
print("Fixed classes:", SELECTED_4_FREQS)

## 10.1 FBCCA

In [ ]:
fbcca_test_4 = fbcca_predict(
    X_test_4,
    y_test_4,
    SELECTED_4_FREQS,
    FS,
    n_harmonics=N_HARMONICS,
    n_subbands=FBCCA_N_SUBBANDS,
    a=FBCCA_A,
    b=FBCCA_B,
    metadata=meta_test_4,
)

print_metrics(
    fbcca_test_4,
    "FBCCA — 4 classes — held-out subjects",
)

display(per_class_accuracy(fbcca_test_4))

In [ ]:
plot_confusion(
    fbcca_test_4,
    "Zhu2021 dry EEG — held-out subjects — 4-class FBCCA",
)

## 10.2 Subject-level robustness

In [ ]:
subject_accuracy = (
    fbcca_test_4
    .assign(
        correct=lambda d: np.isclose(
            d["true_label"].astype(float),
            d["predicted_label"].astype(float),
            atol=0.01,
        )
    )
    .groupby("subject")["correct"]
    .mean()
    .sort_values(ascending=False)
)

display(subject_accuracy.to_frame("accuracy"))

print("Number of held-out subjects:", len(subject_accuracy))
print("Mean subject accuracy:", f"{subject_accuracy.mean():.2%}")
print("Median subject accuracy:", f"{subject_accuracy.median():.2%}")
print("Worst subject accuracy:", f"{subject_accuracy.min():.2%}")
print("Best subject accuracy:", f"{subject_accuracy.max():.2%}")

## 10.3 CCA comparison on the same held-out four-class problem

In [ ]:
cca_test_4 = cca_predict(
    X_test_4,
    y_test_4,
    SELECTED_4_FREQS,
    FS,
    filter_band=(6, 90),
    n_harmonics=N_HARMONICS,
    metadata=meta_test_4,
)

comparison = pd.DataFrame({
    "model": ["CCA", "FBCCA"],
    "accuracy": [
        safe_accuracy(cca_test_4),
        safe_accuracy(fbcca_test_4),
    ],
    "balanced_accuracy": [
        safe_balanced_accuracy(cca_test_4),
        safe_balanced_accuracy(fbcca_test_4),
    ],
})

display(comparison)

# 11. Final summary

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Selected frequencies",
        "Development subjects",
        "Held-out subjects",
        "Development 12-class FBCCA accuracy",
        "Development selected-subset balanced accuracy",
        "Held-out 4-class FBCCA accuracy",
        "Held-out 4-class FBCCA balanced accuracy",
        "Held-out 4-class CCA accuracy",
    ],
    "value": [
        ", ".join(f"{freq:.2f} Hz" for freq in SELECTED_4_FREQS),
        len(selection_subjects),
        len(test_subjects),
        f"{safe_accuracy(fbcca_selection_12):.2%}",
        f"{selection_subset_results.iloc[0]['balanced_accuracy']:.2%}",
        f"{safe_accuracy(fbcca_test_4):.2%}",
        f"{safe_balanced_accuracy(fbcca_test_4):.2%}",
        f"{safe_accuracy(cca_test_4):.2%}",
    ],
})

display(summary)

The reported four-class generalization result is the performance on the **held-out subject cohort**. Frequency selection is performed exclusively on the development participants, and the selected frequencies are fixed before held-out evaluation. CCA and FBCCA remain calibration-free at the participant level: no cross-subject classifier parameters are learned.

# 12. References

1. Zhu F., Jiang L., Dong G., Gao X., Wang Y.  
   **An Open Dataset for Wearable SSVEP-Based Brain-Computer Interfaces.**  
   *Sensors*. 2021;21(4):1256.  
   https://doi.org/10.3390/s21041256

2. Official Zhu2021 dataset README:  
   https://bci.med.tsinghua.edu.cn/upload/zhufangkun/Readme.pdf

3. Official target-frequency / phase mapping:  
   https://bci.med.tsinghua.edu.cn/upload/zhufangkun/stimulation_information.pdf

4. Figshare dataset:  
   https://figshare.com/articles/dataset/An_Open_Dataset_for_Wearable_SSVEP-Based_Brain-Computer_Interfaces/13560281

5. Chen X. et al.  
   **Filter bank canonical correlation analysis for implementing a high-speed SSVEP-based BCI.**  
   *Journal of Neural Engineering*. 2015;12(4):046008.  
   https://doi.org/10.1088/1741-2560/12/4/046008